# 02 — Supervised Classification
**Owner: Jamie | Status: Waiting on 01**

## What this notebook does
- Loads train/val/test CSVs from data/processed/
- Builds TF-IDF vectors (5000 features): **fit on train** `cleaned_text`, **transform** val and test (same vocabulary)
- Trains two classifiers via OneVsRestClassifier:
  1. MultinomialNB — course method
  2. LogisticRegression class_weight='balanced' — new method
- Computes macro-F1, per-class F1, ROC-AUC, confusion matrix
- Saves trained models to data/models/

## Input files (already in repo — just pull)
- data/processed/train.csv
- data/processed/val.csv
- data/processed/test.csv

## Output files (committed to GitHub after this runs)
- data/models/tfidf.pkl
- data/models/nb_model.pkl
- data/models/lr_model.pkl

## How to run
Pull latest from GitHub. Run cells top to bottom.
DO NOT refit TF-IDF on val or test — train only.

In [ ]:
# Import libraries
from pathlib import Path

import pandas as pd
import numpy as np
import re
import nltk
from better_profanity import profanity
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Download NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load splits from data/processed (run notebook from notebooks/ so ROOT resolves like 01_data_prep)
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

[nltk_data] Downloading package stopwords to /Users/kaffe/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/kaffe/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/kaffe/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
# ----------------------
# Text Preprocessing
# ----------------------
# Initialize tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
profanity.load_censor_words()


def preprocess_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove URLs (Reddit links)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    # 3. Remove special characters, numbers, and extra spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    # 4. Tokenize, remove stopwords, lemmatize, then drop profane tokens
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    # drop profane tokens
    # -------------DISSCUSSION-------------
    # maybe we shouldn't filter out profanity, because it represent some emotions like anger，disappointment
    # tokens = [t for t in tokens if not profanity.contains_profanity(t)]
    # -------------DISSCUSSION-------------

    # 5. Join tokens back to text
    return " ".join(tokens)



for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["cleaned_text"] = df["text"].apply(preprocess_text)

print("Preprocessing done for train, val, test.")
print(f"  train: {len(train_df)} rows | val: {len(val_df)} rows | test: {len(test_df)} rows")

label_cols = [c for c in train_df.columns if c not in ("text", "cleaned_text")]
print("\nSample (train):")
print(train_df[["text", "cleaned_text"] + label_cols].head(3))

Preprocessing done for train, val, test.
  train: 43410 rows | val: 5426 rows | test: 5427 rows

Sample (train):
                                                text  \
0  My favourite food is anything I didn't have to...   
1  Now if he does off himself, everyone will thin...   
2                     WHY THE FUCK IS BAYLESS ISOING   

                                        cleaned_text  anger  disgust  fear  \
0                 favourite food anything didnt cook      0        0     0   
1  everyone think he laugh screwing people instea...      0        0     0   
2                                fuck bayless isoing      1        0     0   

   joy  sadness  surprise  neutral  
0    0        0         0        1  
1    0        0         0        1  
2    0        0         0        0  


In [ ]:
# ----------------------
# TF-IDF Vectorization
# ----------------------
# Fit on train only; transform val/test with the same vocabulary (no leakage).
label_cols = [c for c in train_df.columns if c not in ("text", "cleaned_text")]

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
)

X_train = tfidf_vectorizer.fit_transform(train_df["cleaned_text"])
X_val = tfidf_vectorizer.transform(val_df["cleaned_text"])
X_test = tfidf_vectorizer.transform(test_df["cleaned_text"])

y_train = train_df[label_cols].to_numpy()
y_val = val_df[label_cols].to_numpy()
y_test = test_df[label_cols].to_numpy()

print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"X_train: {X_train.shape} (sparse) | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} (sparse) | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape} (sparse) | y_test:  {y_test.shape}")
# print(X_train)
# print(y_train)

Vocabulary size: 5000
X_train: (43410, 5000) (sparse) | y_train: (43410, 7)
X_val:   (5426, 5000) (sparse) | y_val:   (5426, 7)
X_test:  (5427, 5000) (sparse) | y_test:  (5427, 7)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 215287 stored elements and shape (43410, 5000)>
  Coords	Values
  (0, 1456)	0.5324152961465871
  (0, 1550)	0.4676929040509432
  (0, 1065)	0.3339781035944106
  (0, 833)	0.6214949124884436
  (1, 4416)	0.257461932282282
  (1, 2354)	0.3943371567750086
  (1, 3757)	0.5488132270987873
  (1, 3103)	0.2478481922651905
  (1, 2155)	0.39392053890193246
  (1, 41)	0.3051921835358624
  (1, 968)	0.4089880385832717
  (2, 1609)	1.0
  (3, 2623)	0.4479734292052376
  (3, 1466)	0.4879945217263657
  (3, 2627)	0.7491202530242761
  (4, 1101)	0.675381784424522
  (4, 4007)	0.7374682672953112
  (5, 2987)	0.2754916029719183
  (5, 2200)	0.23523147675312903
  (5, 1718)	0.18148207952435144
  (5, 1939)	0.23832492873155378
  (5, 3194)	0.3266871463366536
  (5, 1239)	0.352500376975948